In [1]:
!pip install tensorly --quiet
!pip install timm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 73.1 MB/s eta 0:00:00:00:010:01


In [1]:
"""
================================================================================
NOTEBOOK 0 of 9: DATASET PREPARATION & EXPLORATION  (FIXED v3)
================================================================================

VERIFIED FILE PATHS (confirmed from your Kaggle setup):

APTOS:
  /kaggle/input/aptos2019-blindness-detection/train.csv
  /kaggle/input/aptos2019-blindness-detection/train_images/*.png

DDR (mariaherrerot):
  /kaggle/input/datasets/mariaherrerot/ddrdataset/DR_grading.csv
  /kaggle/input/datasets/mariaherrerot/ddrdataset/DR_grading/DR_grading/*.jpg

IDRiD (aaryapatel98) — Section B ONLY:
  /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/
    B.%20Disease%20Grading/B. Disease Grading/1. Original Images/a. Training Set/*.jpg
    B.%20Disease%20Grading/B. Disease Grading/1. Original Images/b. Testing Set/*.jpg
    B.%20Disease%20Grading/B. Disease Grading/2. Groundtruths/
      a. IDRiD_Disease Grading_Training Labels.csv
      b. IDRiD_Disease Grading_Testing Labels.csv

SKIPPED (duplicates aaryapatel98 IDRiD):
  /kaggle/input/datasets/mariaherrerot/idrid-dataset/  ← SKIP

GPU:     None (CPU only)
Runtime: ~20-35 minutes

AFTER RUNNING:
  Save Version → Save & Run All → Output tab → New Dataset → "unified-dr-dataset"
  Then add "unified-dr-dataset" as input to Notebooks 1-8.

Expected output: ~13,000+ images (vs 3,553 from APTOS-only)
================================================================================
"""

import os
import sys
import glob
import json
import hashlib
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from pathlib import Path
from typing import List, Tuple, Dict, Optional, Set
from collections import Counter, defaultdict
from tqdm.auto import tqdm
import matplotlib
matplotlib.use('Agg')          # headless-safe
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── pretty header ─────────────────────────────────────────────────────────────
print("=" * 80)
print("🔬 NOTEBOOK 0 v3: DATASET PREPARATION & EXPLORATION")
print("=" * 80)

# =============================================================================
# HARD-CODED VERIFIED PATHS
# =============================================================================

# ── APTOS ─────────────────────────────────────────────────────────────────────
APTOS_CSV        = "/kaggle/input/aptos2019-blindness-detection/train.csv"
APTOS_IMG_DIR    = "/kaggle/input/aptos2019-blindness-detection/train_images"

# ── DDR (mariaherrerot) ───────────────────────────────────────────────────────
DDR_CSV          = "/kaggle/input/datasets/mariaherrerot/ddrdataset/DR_grading.csv"
DDR_IMG_DIR      = "/kaggle/input/datasets/mariaherrerot/ddrdataset/DR_grading/DR_grading"

# ── IDRiD (aaryapatel98) — Section B Disease Grading ONLY ────────────────────
_IDRID_BASE      = ("/kaggle/input/datasets/aaryapatel98"
                    "/indian-diabetic-retinopathy-image-dataset"
                    "/B.%20Disease%20Grading/B. Disease Grading")

IDRID_TRAIN_DIR  = os.path.join(_IDRID_BASE, "1. Original Images", "a. Training Set")
IDRID_TEST_DIR   = os.path.join(_IDRID_BASE, "1. Original Images", "b. Testing Set")
IDRID_TRAIN_CSV  = os.path.join(_IDRID_BASE, "2. Groundtruths",
                                 "a. IDRiD_Disease Grading_Training Labels.csv")
IDRID_TEST_CSV   = os.path.join(_IDRID_BASE, "2. Groundtruths",
                                 "b. IDRiD_Disease Grading_Testing Labels.csv")

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR    = "/kaggle/working"
OUTPUT_CSV    = os.path.join(OUTPUT_DIR, "unified_dataset.csv")
STATS_JSON    = os.path.join(OUTPUT_DIR, "dataset_stats.json")
VIZ_DIR       = os.path.join(OUTPUT_DIR, "preprocessed_data")
os.makedirs(VIZ_DIR, exist_ok=True)

# ── Quality-filter thresholds ─────────────────────────────────────────────────
MIN_IMAGE_SIZE  = 100     # pixels
MIN_CONTRAST    = 15.0    # grayscale std  (lowered from 20 — IDRiD images can be darker)
MAX_BLACK_RATIO = 0.92    # fraction of near-black pixels

# ── DDR: grade 5 = ungradable → exclude ──────────────────────────────────────
DDR_EXCLUDE_GRADES: Set[int] = {5}

# ── Human-readable grade names ────────────────────────────────────────────────
GRADE_NAMES = {
    0: "No DR",
    1: "Mild NPDR",
    2: "Moderate NPDR",
    3: "Severe NPDR",
    4: "Proliferative DR",
}


# =============================================================================
# STEP 0 — DIAGNOSTIC: print exactly what Kaggle has mounted
# =============================================================================

def run_diagnostics():
    """Print the file-system tree so we can verify paths before loading."""
    print("\n" + "=" * 70)
    print("STEP 0: DIAGNOSTICS — verifying mounted paths")
    print("=" * 70)

    checks = {
        "APTOS CSV"         : APTOS_CSV,
        "APTOS img dir"     : APTOS_IMG_DIR,
        "DDR CSV"           : DDR_CSV,
        "DDR img dir"       : DDR_IMG_DIR,
        "IDRiD train dir"   : IDRID_TRAIN_DIR,
        "IDRiD test dir"    : IDRID_TEST_DIR,
        "IDRiD train CSV"   : IDRID_TRAIN_CSV,
        "IDRiD test CSV"    : IDRID_TEST_CSV,
    }

    all_ok = True
    for label, path in checks.items():
        exists = os.path.exists(path)
        status = "✅" if exists else "❌"
        if not exists:
            all_ok = False
        # count items if directory
        extra = ""
        if exists and os.path.isdir(path):
            n = sum(1 for f in os.listdir(path)
                    if f.lower().endswith((".jpg", ".jpeg", ".png")))
            extra = f"  ({n:,} images)"
        if exists and os.path.isfile(path) and path.endswith(".csv"):
            try:
                df = pd.read_csv(path, nrows=0)
                extra = f"  columns={list(df.columns)}"
            except Exception as e:
                extra = f"  (CSV read error: {e})"
        print(f"  {status} {label:20s}  {path}{extra}")

    if not all_ok:
        print("\n⚠️  Some paths not found — the loaders will skip missing datasets.")
        print("   Check that all 3 Kaggle datasets are added to this notebook.")
    else:
        print("\n✅ All paths verified!")

    # Also print /kaggle/input tree (1 level)
    print("\n/kaggle/input/ (top level):")
    for item in sorted(os.listdir("/kaggle/input")):
        full = os.path.join("/kaggle/input", item)
        n = _count_images(full)
        print(f"  {'📁' if os.path.isdir(full) else '📄'} {item}  ({n:,} imgs)" )

    # datasets sub-level
    ds_root = "/kaggle/input/datasets"
    if os.path.exists(ds_root):
        print("\n/kaggle/input/datasets/ (two levels):")
        for user in sorted(os.listdir(ds_root)):
            user_path = os.path.join(ds_root, user)
            if not os.path.isdir(user_path):
                continue
            for ds in sorted(os.listdir(user_path)):
                ds_path = os.path.join(user_path, ds)
                n = _count_images(ds_path)
                print(f"  📁 {user}/{ds}  ({n:,} imgs)")


def _count_images(path: str) -> int:
    count = 0
    try:
        for _, _, files in os.walk(path):
            count += sum(1 for f in files
                         if f.lower().endswith((".jpg", ".jpeg", ".png")))
    except PermissionError:
        pass
    return count


# =============================================================================
# STEP 1 — APTOS 2019
# =============================================================================

def load_aptos() -> List[Tuple]:
    """
    APTOS 2019 Blindness Detection.
    CSV : train.csv  — columns: id_code, diagnosis (0-4)
    Imgs: train_images/<id_code>.png
    """
    print("\n" + "=" * 60)
    print("📊 LOADING: APTOS 2019")
    print("=" * 60)

    samples: List[Tuple] = []

    if not os.path.exists(APTOS_CSV):
        print("  ❌ CSV not found — skipping APTOS"); return samples
    if not os.path.exists(APTOS_IMG_DIR):
        print("  ❌ Image dir not found — skipping APTOS"); return samples

    df = pd.read_csv(APTOS_CSV)
    print(f"  CSV   : {APTOS_CSV}  ({len(df):,} rows)")
    print(f"  Cols  : {list(df.columns)}")
    print(f"  ImgDir: {APTOS_IMG_DIR}")

    # Detect columns robustly
    img_col   = _find_col(df, ["id_code", "image", "name", "id", "file"])
    grade_col = _find_col(df, ["diagnosis", "grade", "level", "label", "dr"])

    if img_col is None or grade_col is None:
        print(f"  ❌ Cannot detect columns: img={img_col}, grade={grade_col}")
        return samples

    print(f"  Using: img_col='{img_col}'  grade_col='{grade_col}'")

    not_found = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="  APTOS"):
        code  = str(row[img_col]).strip()
        try:
            grade = int(row[grade_col])
        except (ValueError, TypeError):
            continue
        if not (0 <= grade <= 4):
            continue

        found = False
        for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG"):
            # Remove extension if already present
            stem = code.rsplit(".", 1)[0] if "." in code else code
            p = os.path.join(APTOS_IMG_DIR, stem + ext)
            if os.path.exists(p):
                samples.append((p, int(grade > 0), grade, "APTOS"))
                found = True
                break
        if not found:
            not_found += 1

    print(f"  ✅ Loaded    : {len(samples):,}")
    if not_found:
        print(f"  ⚠️  Not found: {not_found:,}")
    _print_grade_table(samples)
    return samples


# =============================================================================
# STEP 2 — DDR  (CSV-based, flat image directory)
# =============================================================================

def load_ddr() -> List[Tuple]:
    """
    DDR dataset (mariaherrerot).
    CSV : DR_grading.csv  — columns vary, auto-detected
    Imgs: DR_grading/DR_grading/*.jpg  (ALL images in ONE flat folder)
    Grade 5 = ungradable → EXCLUDED.
    """
    print("\n" + "=" * 60)
    print("📊 LOADING: DDR  (grade-5 excluded)")
    print("=" * 60)

    samples: List[Tuple] = []

    if not os.path.exists(DDR_CSV):
        print("  ❌ CSV not found — skipping DDR"); return samples
    if not os.path.exists(DDR_IMG_DIR):
        print("  ❌ Image dir not found — skipping DDR"); return samples

    # ── Read CSV ──────────────────────────────────────────────────────────────
    df = pd.read_csv(DDR_CSV, header=None)    # try no-header first
    print(f"  CSV raw shape: {df.shape},  first row: {list(df.iloc[0])}")

    # Detect if first row is a header
    first_row = [str(v).strip().lower() for v in df.iloc[0]]
    has_header = any(k in first_row for k in
                     ("image", "id", "diagnosis", "grade", "label", "name"))
    if has_header:
        df = pd.read_csv(DDR_CSV)
        print(f"  Detected header: {list(df.columns)}")
    else:
        # No header — assign names based on content
        # Typical DDR: col0=filename, col1=grade
        df.columns = [f"col{i}" for i in range(len(df.columns))]
        print(f"  No header detected. Columns: {list(df.columns)}")

    # ── Identify filename and grade columns ───────────────────────────────────
    img_col   = None
    grade_col = None

    for col in df.columns:
        sample_vals = df[col].dropna().astype(str).head(20).tolist()
        # Filename column: values look like filenames (contain "." or "-")
        looks_like_file = sum(
            1 for v in sample_vals
            if "." in v or "-" in v or "_" in v
        ) >= len(sample_vals) * 0.6
        # Grade column: values are integers 0-5
        try:
            numeric_vals = pd.to_numeric(df[col].dropna().head(50), errors='coerce')
            looks_like_grade = (
                numeric_vals.notna().all() and
                numeric_vals.between(0, 5).all() and
                numeric_vals.nunique() <= 7
            )
        except Exception:
            looks_like_grade = False

        if img_col is None and looks_like_file:
            img_col = col
        if grade_col is None and looks_like_grade and col != img_col:
            grade_col = col

    # Fallback: first two columns
    if img_col is None:
        img_col = df.columns[0]
    if grade_col is None:
        grade_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]

    print(f"  Using: img_col='{img_col}'  grade_col='{grade_col}'")
    print(f"  Grade distribution in CSV:")
    try:
        print(f"  {df[grade_col].value_counts().sort_index().to_dict()}")
    except Exception:
        pass

    # ── Build image filename index for fast lookup ─────────────────────────────
    print(f"  Building image index from {DDR_IMG_DIR} ...")
    img_index: Dict[str, str] = {}   # stem_lower → full_path
    for fname in os.listdir(DDR_IMG_DIR):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            stem = fname.rsplit(".", 1)[0].lower()
            img_index[stem] = os.path.join(DDR_IMG_DIR, fname)
    print(f"  Image index size: {len(img_index):,}")

    # ── Match CSV rows to images ───────────────────────────────────────────────
    excluded_g5 = 0
    not_found   = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="  DDR"):
        img_name = str(row[img_col]).strip()
        try:
            grade = int(float(row[grade_col]))
        except (ValueError, TypeError):
            continue

        # Skip grade 5 (ungradable)
        if grade in DDR_EXCLUDE_GRADES:
            excluded_g5 += 1
            continue
        if not (0 <= grade <= 4):
            continue

        # Try exact stem match
        stem = img_name.rsplit(".", 1)[0].lower()
        if stem in img_index:
            p = img_index[stem]
            samples.append((p, int(grade > 0), grade, "DDR"))
        else:
            # Try with extension already in name
            if img_name.lower() in img_index:
                p = img_index[img_name.lower()]
                samples.append((p, int(grade > 0), grade, "DDR"))
            else:
                not_found += 1

    print(f"  ✅ Loaded         : {len(samples):,}")
    print(f"  ⚠️  Grade-5 excl  : {excluded_g5:,}")
    if not_found:
        print(f"  ⚠️  Not found     : {not_found:,}")
    _print_grade_table(samples)
    return samples


# =============================================================================
# STEP 3 — IDRiD  (aaryapatel98 — Section B Disease Grading ONLY)
# =============================================================================

def load_idrid_aaryapatel98() -> List[Tuple]:
    """
    IDRiD — Indian Diabetic Retinopathy Image Dataset (aaryapatel98 version).
    Uses ONLY Section B (Disease Grading).
    Loads both Training Set and Testing Set with official CSV labels.

    CSV columns: 'Image name'  (e.g. IDRiD_001, NO extension)
                 'Retinopathy grade'  (0-4)
                 'Risk of macular edema'  (ignored)

    IMPORTANT: Training and Testing both have IDRiD_001.jpg etc.
               They are DIFFERENT images — differentiated by their directory.
               We tag them 'IDRiD_train' vs 'IDRiD_test' for dedup safety.
    """
    print("\n" + "=" * 60)
    print("📊 LOADING: IDRiD — aaryapatel98 (Section B only)")
    print("=" * 60)

    samples: List[Tuple] = []

    # Verify paths
    for label, path in [
        ("Train dir",  IDRID_TRAIN_DIR),
        ("Test dir",   IDRID_TEST_DIR),
        ("Train CSV",  IDRID_TRAIN_CSV),
        ("Test CSV",   IDRID_TEST_CSV),
    ]:
        exists = "✅" if os.path.exists(path) else "❌"
        print(f"  {exists} {label}: {path}")

    splits = [
        ("train", IDRID_TRAIN_DIR, IDRID_TRAIN_CSV),
        ("test",  IDRID_TEST_DIR,  IDRID_TEST_CSV),
    ]

    for split_name, img_dir, csv_path in splits:
        if not os.path.exists(img_dir):
            print(f"  ⚠️  {split_name} image dir missing — skip"); continue
        if not os.path.exists(csv_path):
            print(f"  ⚠️  {split_name} CSV missing — skip"); continue

        # ── Read CSV ──────────────────────────────────────────────────────────
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"  ❌ Cannot read {csv_path}: {e}"); continue

        print(f"\n  [{split_name.upper()}] CSV: {os.path.basename(csv_path)}")
        print(f"    Rows  : {len(df)}")
        print(f"    Cols  : {list(df.columns)}")

        # Detect columns
        img_col   = _find_col(df, ["image name", "image_name", "id", "name", "file"])
        grade_col = _find_col(df, ["retinopathy grade", "retinopathy_grade",
                                    "grade", "diagnosis", "level", "label"])

        if img_col is None or grade_col is None:
            # Last-resort: first two columns
            img_col   = df.columns[0]
            grade_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]
            print(f"    ⚠️  Auto-fallback: img='{img_col}' grade='{grade_col}'")
        else:
            print(f"    Using: img='{img_col}'  grade='{grade_col}'")

        # ── Build image index for this split's directory ───────────────────────
        img_index: Dict[str, str] = {}
        for fname in os.listdir(img_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                stem = fname.rsplit(".", 1)[0].lower()
                img_index[stem] = os.path.join(img_dir, fname)

        print(f"    Images in dir: {len(img_index):,}")

        # ── Match CSV rows ──────────────────────────────────────────────────────
        loaded    = 0
        not_found = 0

        for _, row in tqdm(df.iterrows(), total=len(df),
                           desc=f"  IDRiD-{split_name}"):
            img_name = str(row[img_col]).strip()
            try:
                grade = int(float(row[grade_col]))
            except (ValueError, TypeError):
                continue
            if not (0 <= grade <= 4):
                continue

            # Remove extension if present
            stem = img_name.rsplit(".", 1)[0].lower()

            if stem in img_index:
                p = img_index[stem]
                # Tag with split so dedup key is unique
                tag = f"IDRiD_{split_name}"
                samples.append((p, int(grade > 0), grade, tag))
                loaded += 1
            else:
                not_found += 1

        print(f"    ✅ Loaded    : {loaded:,}")
        if not_found:
            print(f"    ⚠️  Not found: {not_found:,}")

    if samples:
        _print_grade_table(samples)
    else:
        print("  ❌ No IDRiD samples loaded")

    return samples


# =============================================================================
# HELPER: column detector
# =============================================================================

def _find_col(df: pd.DataFrame, keywords: List[str]) -> Optional[str]:
    """
    Find the first column whose name (lowercased) contains any of the keywords.
    Tries exact match first, then partial match.
    """
    col_lower = {c.lower().strip(): c for c in df.columns}

    # Exact match
    for kw in keywords:
        if kw in col_lower:
            return col_lower[kw]

    # Partial match
    for kw in keywords:
        for col_name_lower, col_name_orig in col_lower.items():
            if kw in col_name_lower:
                return col_name_orig

    return None


def _print_grade_table(samples: List[Tuple]):
    """Print grade distribution."""
    grades = Counter(s[2] for s in samples)
    total  = len(samples)
    for g in sorted(grades):
        name = GRADE_NAMES.get(g, f"Grade {g}")
        c    = grades[g]
        print(f"    Grade {g} ({name:17s}): {c:6,}  ({c/total*100:5.1f}%)")


# =============================================================================
# STEP 4 — DEDUPLICATION
# =============================================================================

def deduplicate(samples: List[Tuple]) -> List[Tuple]:
    """
    Remove duplicates using (dataset_tag, stem) as key.

    Strategy:
      - Within same dataset tag: keep first occurrence (path-based dedup)
      - Across APTOS/DDR: no overlap expected (different naming conventions)
      - IDRiD_train vs IDRiD_test: different tags → no dedup between them ✅
      - mariaherrerot IDRiD is already SKIPPED → no cross-source dedup needed

    Secondary: path-level dedup catches any remaining duplicates.
    """
    print("\n" + "=" * 60)
    print("STEP 4: DEDUPLICATION")
    print("=" * 60)
    print(f"  Input: {len(samples):,} samples")

    seen_paths: Set[str] = set()
    seen_keys:  Set[str] = set()
    result: List[Tuple]  = []
    dup_paths = 0
    dup_keys  = 0

    for s in samples:
        path  = s[0]
        tag   = s[3]
        stem  = Path(path).stem.lower()
        key   = f"{tag}::{stem}"

        # Level 1: exact path duplicate
        if path in seen_paths:
            dup_paths += 1
            continue

        # Level 2: (tag, stem) duplicate
        if key in seen_keys:
            dup_keys += 1
            continue

        seen_paths.add(path)
        seen_keys.add(key)
        result.append(s)

    print(f"  Removed exact-path duplicates : {dup_paths:,}")
    print(f"  Removed (tag,stem) duplicates : {dup_keys:,}")
    print(f"  Output: {len(result):,} samples")

    # Normalise tag: merge IDRiD_train + IDRiD_test → IDRiD
    result = [
        (s[0], s[1], s[2], s[3].replace("IDRiD_train", "IDRiD").replace("IDRiD_test", "IDRiD"))
        for s in result
    ]
    return result


# =============================================================================
# STEP 5 — QUALITY FILTERING
# =============================================================================

def filter_quality(samples: List[Tuple]) -> List[Tuple]:
    """
    Reject images that are:
      • corrupt / unreadable by OpenCV
      • smaller than MIN_IMAGE_SIZE on either axis
      • very low contrast  (grayscale std < MIN_CONTRAST)
      • mostly black       (> MAX_BLACK_RATIO dark pixels)
    """
    print("\n" + "=" * 60)
    print("STEP 5: QUALITY FILTERING")
    print("=" * 60)
    print(f"  min_size    = {MIN_IMAGE_SIZE}px")
    print(f"  min_contrast= {MIN_CONTRAST} (grayscale std)")
    print(f"  max_black   = {MAX_BLACK_RATIO}")
    print(f"  Input: {len(samples):,} samples")

    passed:  List[Tuple] = []
    reasons: Dict[str, int] = {
        "corrupt": 0, "small": 0, "low_contrast": 0, "mostly_black": 0
    }

    for s in tqdm(samples, desc="  Filtering"):
        path = s[0]
        try:
            img = cv2.imread(path)
            if img is None:
                reasons["corrupt"] += 1; continue

            h, w = img.shape[:2]
            if h < MIN_IMAGE_SIZE or w < MIN_IMAGE_SIZE:
                reasons["small"] += 1; continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            if gray.std() < MIN_CONTRAST:
                reasons["low_contrast"] += 1; continue

            if (gray < 10).sum() / gray.size > MAX_BLACK_RATIO:
                reasons["mostly_black"] += 1; continue

            passed.append(s)

        except Exception:
            reasons["corrupt"] += 1

    total_bad = sum(reasons.values())
    print(f"\n  ✅ Passed  : {len(passed):,}  ({len(passed)/len(samples)*100:.1f}%)")
    print(f"  ❌ Rejected: {total_bad:,}")
    for k, v in reasons.items():
        if v > 0:
            print(f"      {k:15s}: {v:,}")

    return passed


# =============================================================================
# STEP 6 — BUILD DATAFRAME & STATISTICS
# =============================================================================

def build_dataframe_and_stats(samples: List[Tuple]):
    """Build unified DataFrame and compute statistics."""
    print("\n" + "=" * 60)
    print("STEP 6: BUILD UNIFIED DATAFRAME")
    print("=" * 60)

    df = pd.DataFrame(
        samples,
        columns=["image_path", "binary_label", "original_label", "dataset"]
    )
    df["binary_label"]   = df["binary_label"].astype(int)
    df["original_label"] = df["original_label"].astype(int)

    # Shuffle for good measure
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    # ── Statistics ────────────────────────────────────────────────────────────
    n_neg = int((df["binary_label"] == 0).sum())
    n_pos = int((df["binary_label"] == 1).sum())
    pos_weight = round(n_neg / max(n_pos, 1), 4)

    stats = {
        "total":        len(df),
        "n_datasets":   int(df["dataset"].nunique()),
        "n_neg":        n_neg,
        "n_pos":        n_pos,
        "pos_weight":   pos_weight,
        "per_dataset":  {},
        "grade_counts": df["original_label"].value_counts().sort_index().to_dict(),
    }

    for ds in sorted(df["dataset"].unique()):
        sub = df[df["dataset"] == ds]
        stats["per_dataset"][ds] = {
            "total":   int(len(sub)),
            "no_dr":   int((sub["binary_label"] == 0).sum()),
            "dr":      int((sub["binary_label"] == 1).sum()),
            "dr_pct":  round((sub["binary_label"] == 1).mean() * 100, 1),
        }

    # ── Print summary ─────────────────────────────────────────────────────────
    print(f"\n{'─'*55}")
    print(f"  UNIFIED DATASET — {stats['total']:,} samples from "
          f"{stats['n_datasets']} source(s)")
    print(f"{'─'*55}")
    print(f"  {'Dataset':15s}  {'Total':>7}  {'No DR':>7}  {'DR':>7}  {'DR%':>6}")
    print(f"  {'─'*53}")
    for ds, info in stats["per_dataset"].items():
        print(f"  {ds:15s}  {info['total']:7,}  {info['no_dr']:7,}"
              f"  {info['dr']:7,}  {info['dr_pct']:5.1f}%")
    print(f"  {'─'*53}")
    print(f"  {'TOTAL':15s}  {stats['total']:7,}  {stats['n_neg']:7,}"
          f"  {stats['n_pos']:7,}  {stats['n_pos']/stats['total']*100:5.1f}%")
    print(f"\n  pos_weight = {pos_weight:.4f}  (for BCEWithLogitsLoss in NB1–8)")

    print(f"\n  Grade breakdown:")
    for g, c in sorted(stats["grade_counts"].items()):
        name = GRADE_NAMES.get(g, f"Grade {g}")
        print(f"    Grade {g} ({name:17s}): {c:6,}  ({c/stats['total']*100:4.1f}%)")

    return df, stats


# =============================================================================
# STEP 7 — VISUALIZATIONS
# =============================================================================

def create_visualizations(df: pd.DataFrame, stats: Dict, save_dir: str):
    """Publication-quality 8-panel exploration figure (saved headless)."""
    print(f"\n📊 Creating visualizations …")

    # ── Colour palette per dataset ────────────────────────────────────────────
    palette = {
        "APTOS":   "#2E86AB",
        "DDR":     "#A23B72",
        "IDRiD":   "#F18F01",
    }

    fig = plt.figure(figsize=(24, 16))
    gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.44, wspace=0.36)

    ds_order = sorted(df["dataset"].unique())

    # ── (a) Samples per dataset ───────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    ds_counts = df["dataset"].value_counts().reindex(ds_order)
    cols = [palette.get(d, "#888") for d in ds_counts.index]
    bars = ax.bar(range(len(ds_counts)), ds_counts.values,
                  color=cols, edgecolor="black", linewidth=1.5, alpha=0.88)
    ax.set_xticks(range(len(ds_counts)))
    ax.set_xticklabels(ds_counts.index, rotation=20, ha="right", fontsize=11)
    ax.set_ylabel("Count", fontweight="bold")
    ax.set_title("(a) Samples per Dataset", fontweight="bold", fontsize=12, loc="left")
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    for bar, v in zip(bars, ds_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 30,
                f"{v:,}", ha="center", va="bottom", fontweight="bold", fontsize=10)

    # ── (b) Binary label distribution ─────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 1])
    bin_c = df["binary_label"].value_counts().sort_index()
    bars  = ax.bar(["No DR (0)", "DR (1)"], bin_c.values,
                   color=["#4CAF50", "#F44336"],
                   edgecolor="black", linewidth=1.5, alpha=0.88)
    ax.set_ylabel("Count", fontweight="bold")
    ax.set_title("(b) Binary Labels", fontweight="bold", fontsize=12, loc="left")
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    for bar, v in zip(bars, bin_c.values):
        pct = v / len(df) * 100
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 30,
                f"{v:,}\n({pct:.1f}%)", ha="center", va="bottom",
                fontweight="bold", fontsize=11)

    # ── (c) DR grade distribution ─────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 2])
    gr = df["original_label"].value_counts().sort_index()
    g_colors = ["#4CAF50", "#8BC34A", "#FFC107", "#FF9800", "#F44336"]
    bars = ax.bar([f"G{g}" for g in gr.index], gr.values,
                  color=[g_colors[min(g, 4)] for g in gr.index],
                  edgecolor="black", linewidth=1.5, alpha=0.88)
    ax.set_ylabel("Count", fontweight="bold")
    ax.set_title("(c) Grade Distribution", fontweight="bold", fontsize=12, loc="left")
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    for bar, v in zip(bars, gr.values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                f"{v:,}", ha="center", va="bottom", fontsize=9)

    # ── (d) Dataset contribution pie ──────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 3])
    ds_counts2 = df["dataset"].value_counts()
    cols2 = [palette.get(d, "#888") for d in ds_counts2.index]
    wedges, texts, autos = ax.pie(
        ds_counts2.values, labels=ds_counts2.index, autopct="%1.1f%%",
        colors=cols2, startangle=90, pctdistance=0.80,
        wedgeprops={"edgecolor": "black", "linewidth": 1.5}
    )
    for a in autos:
        a.set_fontweight("bold"); a.set_fontsize(11)
    ax.set_title("(d) Dataset Share", fontweight="bold", fontsize=12)

    # ── (e) Stacked bar: dataset × binary ─────────────────────────────────────
    ax = fig.add_subplot(gs[1, 0:2])
    ct = pd.crosstab(df["dataset"], df["binary_label"]).reindex(ds_order)
    ct.columns = ["No DR", "DR"]
    ct.plot(kind="bar", stacked=True, ax=ax,
            color=["#4CAF50", "#F44336"], edgecolor="black",
            linewidth=1.5, alpha=0.85)
    ax.set_ylabel("Count", fontweight="bold", fontsize=12)
    ax.set_xlabel("")
    ax.set_title("(e) Label per Dataset", fontweight="bold", fontsize=12, loc="left")
    ax.legend(fontsize=11, frameon=True)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    # ── (f) DR rate per dataset ───────────────────────────────────────────────
    ax = fig.add_subplot(gs[1, 2:4])
    dr_rate = (df.groupby("dataset")["binary_label"].mean() * 100).reindex(ds_order)
    cols3   = [palette.get(d, "#888") for d in dr_rate.index]
    bars    = ax.barh(dr_rate.index, dr_rate.values,
                      color=cols3, edgecolor="black", linewidth=1.5, alpha=0.85)
    ax.set_xlabel("DR-Positive Rate (%)", fontweight="bold", fontsize=12)
    ax.set_title("(f) DR Rate per Dataset", fontweight="bold", fontsize=12, loc="left")
    ax.axvline(50, color="gray", ls="--", alpha=0.5, linewidth=1.5)
    ax.grid(axis="x", alpha=0.3, linestyle="--")
    for bar, v in zip(bars, dr_rate.values):
        ax.text(bar.get_width() + 0.8,
                bar.get_y() + bar.get_height()/2.,
                f"{v:.1f}%", va="center", fontweight="bold", fontsize=11)

    # ── (g) Grade distribution per dataset ────────────────────────────────────
    ax = fig.add_subplot(gs[2, 0:2])
    ct2 = pd.crosstab(df["dataset"], df["original_label"]).reindex(ds_order)
    ct2.plot(kind="bar", stacked=True, ax=ax,
             colormap="RdYlGn_r", edgecolor="black", linewidth=1, alpha=0.85)
    ax.set_ylabel("Count", fontweight="bold", fontsize=12)
    ax.set_xlabel("")
    ax.set_title("(g) Grades per Dataset", fontweight="bold", fontsize=12, loc="left")
    ax.legend(title="Grade", fontsize=9, frameon=True)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3, linestyle="--")

    # ── (h) Summary text panel ────────────────────────────────────────────────
    ax = fig.add_subplot(gs[2, 2:4])
    ax.axis("off")
    lines = [
        "UNIFIED DATASET SUMMARY",
        "─" * 45,
        f"Total Samples  : {stats['total']:,}",
        f"Source datasets: {stats['n_datasets']}",
        "",
    ]
    for ds, info in stats["per_dataset"].items():
        lines.append(f"  {ds:12s}: {info['total']:5,}  (DR: {info['dr_pct']:.1f}%)")
    lines += [
        "",
        f"No DR (neg)  : {stats['n_neg']:,}",
        f"DR    (pos)  : {stats['n_pos']:,}",
        f"pos_weight   : {stats['pos_weight']:.4f}",
        "",
        "Grade breakdown:",
    ]
    for g, c in sorted(stats["grade_counts"].items()):
        lines.append(f"  G{g} {GRADE_NAMES.get(g,'?'):17s}: {c:,}")
    ax.text(0.04, 0.97, "\n".join(lines),
            fontsize=10, family="monospace", va="top",
            transform=ax.transAxes,
            bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.85))

    for fmt in ("png", "pdf"):
        out = os.path.join(save_dir, f"dataset_exploration.{fmt}")
        fig.savefig(out, dpi=200, bbox_inches="tight")
        print(f"  💾 {out}")
    plt.close(fig)

    # ── Sample image grid ─────────────────────────────────────────────────────
    _sample_image_grid(df, save_dir)


def _sample_image_grid(df: pd.DataFrame, save_dir: str, n_cols: int = 5):
    """One random fundus image per dataset × grade."""
    print("  Creating sample image grid …")

    datasets = sorted(df["dataset"].unique())
    grades   = sorted(df["original_label"].unique())[:n_cols]
    n_rows   = len(datasets)

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(3 * n_cols, 3.2 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    np.random.seed(42)
    for i, ds in enumerate(datasets):
        for j, g in enumerate(grades):
            ax = axes[i, j]
            sub = df[(df["dataset"] == ds) & (df["original_label"] == g)]
            if len(sub) > 0:
                row = sub.sample(1, random_state=i*10 + j).iloc[0]
                try:
                    img = Image.open(row["image_path"]).convert("RGB")
                    img = img.resize((224, 224))
                    ax.imshow(img)
                except Exception:
                    ax.text(0.5, 0.5, "Error", ha="center", va="center",
                            transform=ax.transAxes, color="red")
            else:
                ax.set_facecolor("#f0f0f0")
                ax.text(0.5, 0.5, "N/A", ha="center", va="center",
                        transform=ax.transAxes, color="gray", fontsize=10)
            ax.axis("off")
            if i == 0:
                ax.set_title(f"Grade {g}\n{GRADE_NAMES.get(g,'')}",
                             fontsize=9, fontweight="bold")
        axes[i, 0].set_ylabel(ds, fontsize=11, fontweight="bold",
                               rotation=0, labelpad=60, va="center")

    plt.suptitle("Sample Fundus Images — Dataset × DR Grade",
                 fontweight="bold", fontsize=14, y=1.01)
    plt.tight_layout()
    out = os.path.join(save_dir, "sample_images.png")
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"  💾 {out}")
    plt.close(fig)


# =============================================================================
# MAIN
# =============================================================================

def main():

    print(f"\nOutput CSV : {OUTPUT_CSV}")
    print(f"Viz dir    : {VIZ_DIR}")

    # ── Step 0: Diagnostics ───────────────────────────────────────────────────
    run_diagnostics()

    # ── Step 1-3: Load each dataset ───────────────────────────────────────────
    all_samples: List[Tuple] = []

    aptos_samples = load_aptos()
    all_samples.extend(aptos_samples)
    print(f"  Running total: {len(all_samples):,}")

    ddr_samples = load_ddr()
    all_samples.extend(ddr_samples)
    print(f"  Running total: {len(all_samples):,}")

    idrid_samples = load_idrid_aaryapatel98()
    all_samples.extend(idrid_samples)
    print(f"  Running total: {len(all_samples):,}")

    if not all_samples:
        print("\n❌ No images loaded from any dataset!")
        print("   Ensure all 3 datasets are added to this Kaggle notebook.")
        return

    # ── Step 4: Deduplication ─────────────────────────────────────────────────
    all_samples = deduplicate(all_samples)

    # ── Step 5: Quality filtering ─────────────────────────────────────────────
    all_samples = filter_quality(all_samples)

    if not all_samples:
        print("\n❌ All samples rejected by quality filter!"); return

    # ── Step 6: Build DataFrame + Stats ──────────────────────────────────────
    df, stats = build_dataframe_and_stats(all_samples)

    # ── Save CSV ──────────────────────────────────────────────────────────────
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n💾 CSV saved : {OUTPUT_CSV}  ({len(df):,} rows)")

    with open(STATS_JSON, "w") as f:
        json.dump(stats, f, indent=2, default=str)
    print(f"💾 Stats saved: {STATS_JSON}")

    # ── Step 7: Visualizations ────────────────────────────────────────────────
    create_visualizations(df, stats, VIZ_DIR)

    # ── Final checklist ───────────────────────────────────────────────────────
    print(f"""
{'='*80}
✅ DATASET PREPARATION COMPLETE
{'='*80}

📁 Output files:
   {OUTPUT_CSV}
   {STATS_JSON}
   {VIZ_DIR}/dataset_exploration.png / .pdf
   {VIZ_DIR}/sample_images.png

📊 Dataset summary:
   Total samples  : {stats['total']:,}
   No DR (neg)    : {stats['n_neg']:,}
   DR    (pos)    : {stats['n_pos']:,}
   pos_weight     : {stats['pos_weight']:.4f}

   train (~70%)   : {int(stats['total']*0.70):,}
   val   (~15%)   : {int(stats['total']*0.15):,}
   test  (~15%)   : {int(stats['total']*0.15):,}

📋 NEXT STEPS:
   1. Click "Save Version" → "Save & Run All (Commit)"
   2. Wait for completion
   3. Output tab → "New Dataset" → name it: unified-dr-dataset
   4. Add "unified-dr-dataset" as input to Notebooks 1–8

⚙️  Copy these values to Config in NB1–8:
   POS_WEIGHT  = {stats['pos_weight']:.4f}
   (auto-loaded from dataset_stats.json — no manual entry needed)
""")


if __name__ == "__main__":
    main()

🔬 NOTEBOOK 0 v3: DATASET PREPARATION & EXPLORATION

Output CSV : /kaggle/working/unified_dataset.csv
Viz dir    : /kaggle/working/preprocessed_data

STEP 0: DIAGNOSTICS — verifying mounted paths
  ✅ APTOS CSV             /kaggle/input/aptos2019-blindness-detection/train.csv  columns=['id_code', 'diagnosis']
  ✅ APTOS img dir         /kaggle/input/aptos2019-blindness-detection/train_images  (3,662 images)
  ✅ DDR CSV               /kaggle/input/datasets/mariaherrerot/ddrdataset/DR_grading.csv  columns=['id_code', 'diagnosis']
  ✅ DDR img dir           /kaggle/input/datasets/mariaherrerot/ddrdataset/DR_grading/DR_grading  (12,524 images)
  ✅ IDRiD train dir       /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/1. Original Images/a. Training Set  (413 images)
  ✅ IDRiD test dir        /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/1. Original Ima

  APTOS:   0%|          | 0/3662 [00:00<?, ?it/s]

  ✅ Loaded    : 3,662
    Grade 0 (No DR            ):  1,805  ( 49.3%)
    Grade 1 (Mild NPDR        ):    370  ( 10.1%)
    Grade 2 (Moderate NPDR    ):    999  ( 27.3%)
    Grade 3 (Severe NPDR      ):    193  (  5.3%)
    Grade 4 (Proliferative DR ):    295  (  8.1%)
  Running total: 3,662

📊 LOADING: DDR  (grade-5 excluded)
  CSV raw shape: (12523, 2),  first row: ['id_code', 'diagnosis']
  Detected header: ['id_code', 'diagnosis']
  Using: img_col='id_code'  grade_col='diagnosis'
  Grade distribution in CSV:
  {0: 6266, 1: 630, 2: 4477, 3: 236, 4: 913}
  Building image index from /kaggle/input/datasets/mariaherrerot/ddrdataset/DR_grading/DR_grading ...
  Image index size: 12,524


  DDR:   0%|          | 0/12522 [00:00<?, ?it/s]

  ✅ Loaded         : 12,522
  ⚠️  Grade-5 excl  : 0
    Grade 0 (No DR            ):  6,266  ( 50.0%)
    Grade 1 (Mild NPDR        ):    630  (  5.0%)
    Grade 2 (Moderate NPDR    ):  4,477  ( 35.8%)
    Grade 3 (Severe NPDR      ):    236  (  1.9%)
    Grade 4 (Proliferative DR ):    913  (  7.3%)
  Running total: 16,184

📊 LOADING: IDRiD — aaryapatel98 (Section B only)
  ✅ Train dir: /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/1. Original Images/a. Training Set
  ✅ Test dir: /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/1. Original Images/b. Testing Set
  ✅ Train CSV: /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
  ✅ Test CSV: /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-datase

...

  IDRiD-test:   0%|          | 0/103 [00:00<?, ?it/s]

    ✅ Loaded    : 103
    Grade 0 (No DR            ):    168  ( 32.6%)
    Grade 1 (Mild NPDR        ):     25  (  4.8%)
    Grade 2 (Moderate NPDR    ):    168  ( 32.6%)
    Grade 3 (Severe NPDR      ):     93  ( 18.0%)
    Grade 4 (Proliferative DR ):     62  ( 12.0%)
  Running total: 16,700

STEP 4: DEDUPLICATION
  Input: 16,700 samples
  Removed exact-path duplicates : 0
  Removed (tag,stem) duplicates : 0
  Output: 16,700 samples

STEP 5: QUALITY FILTERING
  min_size    = 100px
  min_contrast= 15.0 (grayscale std)
  max_black   = 0.92
  Input: 16,700 samples


  Filtering:   0%|          | 0/16700 [00:00<?, ?it/s]

Corrupt JPEG data: 35 extraneous bytes before marker 0xd9
Corrupt JPEG data: 38 extraneous bytes before marker 0xd9
Corrupt JPEG data: 34 extraneous bytes before marker 0xd9



  ✅ Passed  : 16,652  (99.7%)
  ❌ Rejected: 48
      low_contrast   : 48

STEP 6: BUILD UNIFIED DATAFRAME

───────────────────────────────────────────────────────
  UNIFIED DATASET — 16,652 samples from 3 source(s)
───────────────────────────────────────────────────────
  Dataset            Total    No DR       DR     DR%
  ─────────────────────────────────────────────────────
  APTOS              3,643    1,799    1,844   50.6%
  DDR               12,493    6,263    6,230   49.9%
  IDRiD                516      168      348   67.4%
  ─────────────────────────────────────────────────────
  TOTAL             16,652    8,230    8,422   50.6%

  pos_weight = 0.9772  (for BCEWithLogitsLoss in NB1–8)

  Grade breakdown:
    Grade 0 (No DR            ):  8,230  (49.4%)
    Grade 1 (Mild NPDR        ):  1,023  ( 6.1%)
    Grade 2 (Moderate NPDR    ):  5,618  (33.7%)
    Grade 3 (Severe NPDR      ):    521  ( 3.1%)
    Grade 4 (Proliferative DR ):  1,260  ( 7.6%)

💾 CSV saved : /kaggle/workin